In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-1.3b-instruct")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-coder-1.3b-instruct")
messages=[
    { 'role': 'user', 'content': "write a quick sort algorithm in python."}
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt = True, return_tensors="pt", return_dict = False).to(model.device)
# tokenizer.eos_token_id is the id of <|EOT|> token
outputs = model.generate(inputs, max_new_tokens=512, do_sample=False, top_k=50, top_p=0.95, num_return_sequences=1, eos_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-1.3b-instruct")

chat = [
    {"role": "user", "content": "What's the weather in Lyon?"},
    {"role": "assistant", "content": '{"tool_calls": []}'},
]

# 1. no system message: anything you didn't write is the template's default
print(tok.apply_chat_template(chat, tokenize=False))
print("=" * 40)

# 2. with your own system message: it should replace the default
tk = tok.apply_chat_template([{"role": "system", "content": "\n"}] + chat, tokenize=False)
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-coder-1.3b-instruct")
print(tk)
print("=" * 40)
outputs = model.generate(tk, max_new_tokens=512, do_sample=False, top_k=50, top_p=0.95, num_return_sequences=1, eos_token_id=tok.eos_token_id)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

# 3. the prompt alone, as at generation time: shows where the answer starts
print(tok.apply_chat_template(chat[:1], tokenize=False, add_generation_prompt=True))

<｜begin▁of▁sentence｜>You are an AI programming assistant, utilizing the Deepseek Coder model, developed by Deepseek Company, and you only answer questions related to computer science. For politically sensitive questions, security and privacy issues, and other non-computer science questions, you will refuse to answer
### Instruction:
What's the weather in Lyon?
### Response:
{"tool_calls": []}
<|EOT|>



Loading weights: 100%|██████████| 219/219 [00:00<00:00, 19806.64it/s]
[transformers] The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<｜begin▁of▁sentence｜>
### Instruction:
What's the weather in Lyon?
### Response:
{"tool_calls": []}
<|EOT|>



AttributeError: 'str' object has no attribute 'shape'

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "deepseek-ai/deepseek-coder-1.3b-instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

prompt_text = tok.apply_chat_template(
    # [{"role": "system", "content": "\n"}] + chat,
    chat,
    tokenize=False,
    add_generation_prompt=True,
)
print(prompt_text)
print("=" * 40)

inputs = tok(prompt_text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        eos_token_id=tok.eos_token_id,
    )

print(tok.decode(outputs[0][input_len:], skip_special_tokens=True))

Loading weights: 100%|██████████| 219/219 [00:06<00:00, 35.00it/s]


<｜begin▁of▁sentence｜>
### Instruction:
What's the weather in Lyon?
### Response:
{"tool_calls": []}
<|EOT|>
### Response:

I'm sorry, but as an AI, I don't have real-time capabilities to provide current weather updates. I recommend checking a reliable weather forecast website or app for the most accurate and up-to-date information.

